# LangChain — Simple Agent

**Framework:** LangChain (LCEL)  
**Level:** 01 — Simple  
**Model:** `gemini-2.0-flash` via `langchain-google-genai`

### What You Will Learn
- How to define tools with LangChain's `@tool` decorator
- How to build a tool-calling agent using `create_tool_calling_agent` + `AgentExecutor`
- What LCEL (LangChain Expression Language) is and how chains compose
- How `AgentExecutor` manages the ReAct loop automatically (vs. LangGraph which makes it explicit)

## Concept: LCEL Chain Architecture

```
User Input
    │
    ▼
┌──────────────────────────────────┐
│  ChatPromptTemplate              │  ← formats input + chat history
│  [{system}, {human}, {scratchpad}]│
└──────────────┬───────────────────┘
               │ pipe (|)
               ▼
┌──────────────────────────────────┐
│  LLM with bound tools            │  ← decides: answer or call tool?
│  (gemini-2.0-flash)              │
└──────────────┬───────────────────┘
               │
    ┌──────────┴──────────┐
    │ tool_calls?         │ answer text?
    ▼                     ▼
[Tool executes]    [Final Response]
    │
    └─→ result appended to scratchpad → back to LLM
```

**LCEL (LangChain Expression Language)** uses the `|` pipe operator to compose components:  
`chain = prompt | llm_with_tools | output_parser`  
Each component's output becomes the next component's input.

**`AgentExecutor`** wraps the chain and manages the loop: run → check for tool calls → execute → run again.

**Compare to ADK:** ADK uses `InMemoryRunner` + async events. LangChain wraps the same loop in `AgentExecutor.invoke()`. Both hide the loop from you at the simple level.

## Setup

In [ ]:
import os
from dotenv import load_dotenv

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.agents import AgentExecutor, create_tool_calling_agent

load_dotenv()
assert os.getenv("GOOGLE_API_KEY"), "GOOGLE_API_KEY not set — check your .env file"
print("✓ GOOGLE_API_KEY loaded")

## Tool Definitions

LangChain's `@tool` decorator wraps a Python function into a `BaseTool` object.  
The **function docstring** becomes the tool description the LLM reads.  
Unlike ADK (which uses `Args:` sections), LangChain infers the schema from **type annotations**.

> **Mock tools:** no external API keys needed.

In [ ]:
@tool
def get_weather(city: str) -> dict:
    """Return a mock weather report for the given city.

    Use this tool when the user asks about weather conditions, temperature,
    or climate in a specific city.
    """
    mock_data = {
        "london":    {"condition": "Cloudy", "temp_c": 12, "humidity": 78},
        "new york":  {"condition": "Sunny",  "temp_c": 22, "humidity": 45},
        "bangalore": {"condition": "Rainy",  "temp_c": 25, "humidity": 85},
        "tokyo":     {"condition": "Clear",  "temp_c": 18, "humidity": 60},
    }
    key = city.lower()
    if key in mock_data:
        d = mock_data[key]
        return {"city": city, "condition": d["condition"],
                "temperature_celsius": d["temp_c"], "humidity_percent": d["humidity"]}
    return {"error": f"No data for '{city}'. Try: London, New York, Bangalore, Tokyo."}


@tool
def get_time(city: str) -> dict:
    """Return the current local time for the given city.

    Use this tool when the user asks what time it is in a specific location.
    """
    mock_times = {
        "london":    "14:30 GMT",
        "new york":  "09:30 EST",
        "bangalore": "20:00 IST",
        "tokyo":     "22:30 JST",
    }
    key = city.lower()
    if key in mock_times:
        return {"city": city, "local_time": mock_times[key]}
    return {"error": f"No time data for '{city}'. Try: London, New York, Bangalore, Tokyo."}


tools = [get_weather, get_time]

# Inspect what the LLM sees for a tool
print("Tool name:       ", get_weather.name)
print("Tool description:", get_weather.description)
print("Tool schema:     ", get_weather.args)

## Model + Prompt

The prompt has three parts:
- `system` — the agent's persona and behavior rules
- `human` — the user's query (filled in at runtime via `{input}`)
- `MessagesPlaceholder("agent_scratchpad")` — **required** slot where `AgentExecutor` injects intermediate tool call/result messages

In [ ]:
llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0)

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful travel assistant. Use tools to answer weather and time questions. Be concise and friendly."),
    ("human", "{input}"),
    MessagesPlaceholder("agent_scratchpad"),
])

print("Prompt input variables:", prompt.input_variables)

## Agent Definition

`create_tool_calling_agent` wires the prompt, LLM, and tools into an **LCEL runnable**:  
`prompt | llm.bind_tools(tools) | ToolsAgentOutputParser()`

`AgentExecutor` wraps that runnable and manages the loop:  
invoke → parse → execute tools → invoke again → until no more tool calls.

In [ ]:
agent = create_tool_calling_agent(llm, tools, prompt)
executor = AgentExecutor(agent=agent, tools=tools, verbose=True)
# verbose=True prints each step: the LLM's reasoning, tool calls, and tool results
print("AgentExecutor ready")

## Run the Agent

In [ ]:
# Query 1 — single tool call
query1 = "What's the weather like in Bangalore right now?"
print(f"User: {query1}\n")
result1 = executor.invoke({"input": query1})
print(f"\nFinal answer: {result1['output']}")

In [ ]:
# Query 2 — two tool calls
query2 = "What time is it in Tokyo, and is it a good day to go outside?"
print(f"User: {query2}\n")
result2 = executor.invoke({"input": query2})
print(f"\nFinal answer: {result2['output']}")

In [ ]:
# Try your own
your_query = "What's the weather in London and what time is it there?"
result3 = executor.invoke({"input": your_query})
print(f"Agent: {result3['output']}")

## Key Takeaways

| What You Saw | Why It Matters |
|---|---|
| `@tool` decorator | Wraps function → `BaseTool`; docstring becomes the LLM-visible description |
| `MessagesPlaceholder("agent_scratchpad")` | Required slot; AgentExecutor fills it with tool call/result history |
| `create_tool_calling_agent` | Builds the LCEL chain: `prompt \| llm.bind_tools \| parser` |
| `AgentExecutor` | Manages the ReAct loop; `verbose=True` shows each step |
| `executor.invoke({"input": ...})` | Synchronous call; `.stream()` for streaming |

### LangChain vs ADK (Simple Level)

| Dimension | ADK | LangChain |
|---|---|---|
| Tool definition | Plain function + docstring | `@tool` decorator |
| Agent creation | `Agent(model, tools, instruction)` | `create_tool_calling_agent(llm, tools, prompt)` |
| Runner | `InMemoryRunner` (async) | `AgentExecutor` (sync/stream) |
| Session | `InMemorySessionService` | No session by default |
| Loop visibility | Hidden in runner events | Hidden in executor (visible with `verbose=True`) |

### What Changes at 02-Intermediate
- Adds `ConversationBufferMemory` (multi-turn conversation)
- Adds a third tool, structured output with Pydantic
- Uses `RunnableWithMessageHistory` for session-aware chains

### Next Notebook
[02-Intermediate →](../02-intermediate/agent.ipynb)